# Closing the Loop: Genomic Results Back Onto the EPR Problem List

This is the fifteenth notebook in the series. Every notebook so far has followed an
order *out* to a genomics lab. This one follows a result back the other way, onto the
referring Trust's own EPR problem list - using Epic as the worked example, since it's
one of the EPRs that actually carries SNOMED CT on `Condition`, and since a real (if
still read-only) Epic connection already exists elsewhere in this NW-GMSA ecosystem
([nw-gmsa/julius](https://github.com/nw-gmsa/julius)) to build on.

The shape of the whole round trip:

1. A clinician records a suspected condition on the EPR problem list - SNOMED CT
   coded, if the EPR supports it.
2. That SNOMED concept needs converting to whatever coding the genomic order actually
   needs - `10-histocompatibility-immunogenetics-hl7v2-nw-standard.ipynb`,
   `13-snomed-to-hpo-genomic-clinical-indication.ipynb`, and
   `14-national-genomic-test-directory-codesystems.ipynb` are all, in different ways,
   about exactly this conversion problem.
3. The order goes out, coded that way.
4. The genomic report comes back with a coded outcome - confirming, refuting, or
   leaving open the original suspected condition.
5. That outcome can update the EPR problem list directly - the same SNOMED concept
   the clinician started with, now with a `verificationStatus` and an evidence trail
   back to the genomic finding.

## NHS guidance says SNOMED CT; not every EPR does it

NHS guidance for problem lists - FHIR `Condition`, HL7 v2 `DG1` - recommends SNOMED CT
coding. In practice that's inconsistent: solidly established in UK GP systems, and
present in secondary care EPRs like Epic, but far from universal across every Trust
system this repo's other notebooks touch (`Clatterbridge-Order.txt`'s `DG1`-equivalent
fields, back in notebook 10, carried nothing coded at all).

Epic's own FHIR reference for this is
[fhir.epic.com/Specifications?api=953](https://fhir.epic.com/Specifications?api=953) -
worth being upfront about a real limitation before using it: that page (and the two
others this notebook was given -
[api=1075](https://fhir.epic.com/Specifications?api=1075),
[api=949](https://fhir.epic.com/Specifications?api=949)) render their actual content
client-side, keyed to session state a plain fetch doesn't have - both a direct fetch
and a browser-user-agent `curl` got back the same generic "That API was not found"
shell, not the specific specification. Checked below rather than assumed.

In [1]:
import json

import requests

EPIC_PAGES = {
    953: "Condition (Problems) - general reference",
    1075: "Condition example",
    949: "Condition write-back",
}

for api_id, label in EPIC_PAGES.items():
    r = requests.get(f"https://fhir.epic.com/Specifications?api={api_id}")
    not_found = "That API was not found" in r.text
    print(f"api={api_id:<5} ({label:35}) -> HTTP {r.status_code}, generic not-found shell: {not_found}")

api=953   (Condition (Problems) - general reference) -> HTTP 200, generic not-found shell: True


api=1075  (Condition example                  ) -> HTTP 200, generic not-found shell: True


api=949   (Condition write-back               ) -> HTTP 200, generic not-found shell: True


So the `Condition` examples below are built to Epic's well-documented *general* R4
conventions (`category` = `problem-list-item`, `clinicalStatus`/`verificationStatus`
from the standard HL7 Terminology `CodeSystem`s, `code.coding` carrying whichever
terminologies the organisation's clinical content dictionary maps a problem to -
SNOMED CT among them for NHS-configured instances) - illustrative of that shape, not a
verbatim copy of Epic's own numbered example, which this notebook couldn't fetch
live.

## Which code the order actually needs

Genomics doesn't usually order a test *for* a SNOMED CT concept directly - it orders
against a **clinical indication**: the NHS England Genomic Test Directory's own term
for the 1st-level grouping a specific test sits under
(`13-...ipynb`/`14-...ipynb`'s `R*`/`M*` `GenomicClinicalIndication` codes; the new
digital directory's equivalent is a **Test Package (TP)** code -
`14-national-genomic-test-directory-codesystems.ipynb` covers that migration in
depth). For rare disease WGS specifically, Genomics England's own order form uses
something else again - HPO terms, not a clinical-indication code at all. Straight from
the real, current form
([gms-test-order-form-rare-disease-v2.0_April26.pdf](https://www.england.nhs.uk/wp-content/uploads/2021/09/gms-test-order-form-rare-disease-v2.0_April26.pdf)):

> *"HPO terms are NECESSARY for the analysis and interpretation of WGS data and cannot
> commence until provided. Please ensure HPO terms match those available at
> https://hpo.jax.org/app/"* ... *"At least one HPO term is required"*

- the same mandatory-HPO requirement `13-...ipynb` found on
`Questionnaire-GMSWGSRareDisease.html`'s `NOS/HPOTerm` item, now confirmed on the
actual PDF order form too. Either way - clinical indication or HPO - it's a different
code to whatever the EPR's own problem list holds.

## The scenario: suspected Lynch syndrome

A clinician suspects Lynch syndrome and records it on the EPR problem list - SNOMED CT
`716318002` "Lynch syndrome", the same code
`13-snomed-to-hpo-genomic-clinical-indication.ipynb` used throughout (confirmed real
there via `$lookup`), and the same patient (Ned Liverpool) this IG's own published
Lynch syndrome example already uses.

In [2]:
SNOMED_SYSTEM = "http://snomed.info/sct"
LYNCH_SYNDROME_SCT = "716318002"
LYNCH_SYNDROME_DISPLAY = "Lynch syndrome"

epic_style_condition_before = {
    "resourceType": "Condition",
    "id": "example-lynch-suspected",
    "clinicalStatus": {
        "coding": [{"system": "http://terminology.hl7.org/CodeSystem/condition-clinical", "code": "active"}]
    },
    "verificationStatus": {
        "coding": [{"system": "http://terminology.hl7.org/CodeSystem/condition-ver-status", "code": "unconfirmed"}]
    },
    "category": [{
        "coding": [{"system": "http://terminology.hl7.org/CodeSystem/condition-category", "code": "problem-list-item", "display": "Problem List Item"}]
    }],
    "code": {
        "coding": [
            {"system": SNOMED_SYSTEM, "code": LYNCH_SYNDROME_SCT, "display": LYNCH_SYNDROME_DISPLAY},
        ],
        "text": "Suspected Lynch syndrome",
    },
    "subject": {"display": "Ned Liverpool"},
    "recordedDate": "2026-08-01",
}

print(json.dumps(epic_style_condition_before, indent=2))

{
  "resourceType": "Condition",
  "id": "example-lynch-suspected",
  "clinicalStatus": {
    "coding": [
      {
        "system": "http://terminology.hl7.org/CodeSystem/condition-clinical",
        "code": "active"
      }
    ]
  },
  "verificationStatus": {
    "coding": [
      {
        "system": "http://terminology.hl7.org/CodeSystem/condition-ver-status",
        "code": "unconfirmed"
      }
    ]
  },
  "category": [
    {
      "coding": [
        {
          "system": "http://terminology.hl7.org/CodeSystem/condition-category",
          "code": "problem-list-item",
          "display": "Problem List Item"
        }
      ]
    }
  ],
  "code": {
    "coding": [
      {
        "system": "http://snomed.info/sct",
        "code": "716318002",
        "display": "Lynch syndrome"
      }
    ],
    "text": "Suspected Lynch syndrome"
  },
  "subject": {
    "display": "Ned Liverpool"
  },
  "recordedDate": "2026-08-01"
}


## SNOMED CT doesn't get you to `R210` automatically

`13-...ipynb` translated this exact code (`716318002`) against Genomics England's own
public terminology server, live - no mapping to HPO, because it's a *diagnosis*, not
a phenotypic finding. `14-...ipynb` went looking for a `ConceptMap` from SNOMED to the
England Genomic Test Directory / `GenomicClinicalIndication` on that same server and
found none of the four published maps go anywhere close. Both notebooks' findings
point the same way: there is currently no automated SNOMED CT → NHS England genomic
coding route for a diagnosis code, only for the narrower case of an individual
phenotypic sign.

So today this is a manual, curated lookup - the kind
`ClinicalIndication.fsh`/`EnglandTestCode.fsh` are built from directly: **`R210`
"Inherited MMR deficiency (Lynch syndrome)"**. Worth noting, per `14-...ipynb`'s own
`ConceptMap` work: Lynch syndrome hasn't migrated to the new digital directory yet -
there's no `TP` code for it at present, only the legacy `R210`.

In [3]:
GENOMIC_CLINICAL_INDICATION_SYSTEM = "https://fhir.nwgenomics.nhs.uk/CodeSystem/GenomicClinicalIndication"
LYNCH_SYNDROME_R_CODE = "R210"
LYNCH_SYNDROME_R_DISPLAY = "Inherited MMR deficiency (Lynch syndrome)"

print(f"SNOMED CT {LYNCH_SYNDROME_SCT} {LYNCH_SYNDROME_DISPLAY!r}")
print(f"  -> manually cross-referenced to {GENOMIC_CLINICAL_INDICATION_SYSTEM}#{LYNCH_SYNDROME_R_CODE} {LYNCH_SYNDROME_R_DISPLAY!r}")
print("  -> no DGTS Test Package (TP) code yet - still legacy-only")

SNOMED CT 716318002 'Lynch syndrome'
  -> manually cross-referenced to https://fhir.nwgenomics.nhs.uk/CodeSystem/GenomicClinicalIndication#R210 'Inherited MMR deficiency (Lynch syndrome)'
  -> no DGTS Test Package (TP) code yet - still legacy-only


### `R210` on the way out

Once converted, `R210` travels as `OBR-31`/`ServiceRequest.reasonCode` on the
`OML_O21`/FHIR Message `O21` order - the exact field
`10-histocompatibility-immunogenetics-hl7v2-nw-standard.ipynb` traced to
[`hl7v2.html`](https://nw-gmsa.github.io/en/hl7v2.html)'s own worked example:
`R210^Lynch syndrome^GenomicClinicalIndication`. Notebooks `08`-`11` cover building
that order end-to-end; this notebook picks up from the report coming back.

## The report comes back: a coded outcome, not just free text

NW-GMSA already publishes a real Lynch syndrome genomics report example for this exact
patient - fetched live below, not vendored, the same way `14-...ipynb` fetched FSH
source from the public repo (this time the *compiled* FHIR JSON the IG actually
publishes, since there's real structure here worth working with directly).

In [4]:
IG_BASE = "https://nw-gmsa.github.io/en"

diagnostic_report = requests.get(f"{IG_BASE}/DiagnosticReport-DiagnosticReportGenomicsReportLS.json").json()
mutation_finding = requests.get(f"{IG_BASE}/Observation-4490c092-c78c-480a-8cb7-653b70113fd5.json").json()
diagnostic_implication = requests.get(f"{IG_BASE}/Observation-6beb613f-d303-42af-b025-86e8e0872061.json").json()

print("DiagnosticReport.code:      ", [c.get("code") for c in diagnostic_report["code"]["coding"]])
print("DiagnosticReport.conclusion:", repr(diagnostic_report.get("conclusion")))
print("DiagnosticReport.conclusionCode:", diagnostic_report.get("conclusionCode", "<not present>"))
print()
print("Observation (mutation finding).code:", mutation_finding["code"]["coding"])

DiagnosticReport.code:       ['R210.2', '1054161000000101']
DiagnosticReport.conclusion: 'Normal - no action'
DiagnosticReport.conclusionCode: <not present>

Observation (mutation finding).code: [{'system': 'http://snomed.info/sct', 'code': '716318002', 'display': 'Lynch syndrome'}]


`conclusion` is free text ("Normal - no action" in this particular published
example); `conclusionCode` isn't populated at all. That's the gap this scenario's own
`OBX`/LOINC `51968-6` is for - a coded outcome from
[`ValueSet-GenomicTestOutcomeCodes`](https://nw-gmsa.github.io/en/ValueSet-GenomicTestOutcomeCodes.html),
bound to both `DiagnosticReport.conclusionCode` and this IG's own genomic test report
`Questionnaire`. Fetched live, same as any other `CodeSystem` in this series:

In [5]:
outcome_codesystem = requests.get(f"{IG_BASE}/CodeSystem-GenomicTestOutcomeCode.json").json()
print(f"{outcome_codesystem['url']} v{outcome_codesystem['version']} - {len(outcome_codesystem['concept'])} codes")
for concept in outcome_codesystem["concept"][:6]:
    print(" ", concept["code"], concept["display"])
print("  ...")

https://fhir.nwgenomics.nhs.uk/CodeSystem/GenomicTestOutcomeCode v2.2.0 - 23 codes


  311 RESULT CONSISTENT WITH REFERRAL INDICATION
  312 RESULT  PARTIALLY CAUSATIVE OF REFERRAL INDICATION
  313 GENETIC CAUSE WAS NOT FOUND
  314 RESULT OF UNCERTAIN SIGNIFICANCE
  321 VARIANT DETECTED
  322 VARIANT NOT DETECTED
  ...


In [6]:
LYNCH_CONFIRMED_OUTCOME = next(c for c in outcome_codesystem["concept"] if c["code"] == "311")
print(LYNCH_CONFIRMED_OUTCOME)

# The v2 OBX this outcome would travel as - same CE/LOINC shape already real in this
# repo's own ctdna9737383222.txt fixture (there carrying #971 "FAILURE" instead):
obx_line = f'OBX|2|CE|51968-6^^LN|1|{LYNCH_CONFIRMED_OUTCOME["code"]}^{LYNCH_CONFIRMED_OUTCOME["display"]}|||||||||20260801103726+0000'
print()
print(obx_line)

# ...and the DiagnosticReport.conclusionCode this notebook's own published example is
# missing:
diagnostic_report_with_conclusion_code = dict(diagnostic_report)
diagnostic_report_with_conclusion_code["conclusionCode"] = [{
    "coding": [{
        "system": outcome_codesystem["url"],
        "code": LYNCH_CONFIRMED_OUTCOME["code"],
        "display": LYNCH_CONFIRMED_OUTCOME["display"],
    }]
}]
print()
print(json.dumps(diagnostic_report_with_conclusion_code["conclusionCode"], indent=2))

{'code': '311', 'display': 'RESULT CONSISTENT WITH REFERRAL INDICATION'}

OBX|2|CE|51968-6^^LN|1|311^RESULT CONSISTENT WITH REFERRAL INDICATION|||||||||20260801103726+0000

[
  {
    "coding": [
      {
        "system": "https://fhir.nwgenomics.nhs.uk/CodeSystem/GenomicTestOutcomeCode",
        "code": "311",
        "display": "RESULT CONSISTENT WITH REFERRAL INDICATION"
      }
    ]
  }
]


### A more robust alternative: `DiagnosticImplication`

A coded `DiagnosticReport.conclusionCode` is a per-*report* outcome - it doesn't say
*which* suspected condition it's confirming, if a report covers more than one. HL7's
[Diagnostic
Implication](https://build.fhir.org/ig/HL7/genomics-reporting/StructureDefinition-diagnostic-implication.html)
profile (real future direction, not yet in production here, but already modelled as a
published example for this exact patient) is the more precise version of the same
idea - generated by a clinical scientist's review, not just a report-level code:

In [7]:
for component in diagnostic_implication["component"]:
    code_display = component["code"]["coding"][0].get("display", component["code"]["coding"][0]["code"])
    value = component.get("valueCodeableConcept", {})
    value_codings = [(c["system"].rsplit("/", 1)[-1], c["code"], c.get("display")) for c in value.get("coding", [])]
    print(f"{code_display}:")
    for system, vcode, display in value_codings:
        print(f"    {system}#{vcode} {display!r}")
    if value.get("text"):
        print(f"    text: {value['text']!r}")

Genetic variation clinical significance [Imp]:
    loinc.org#LA6668-3 'Pathogenic'
81259-4:
    GenomicClinicalIndication#R210 'Inherited MMR deficiency (Lynch syndrome)'
    sct#716318002 'Lynch syndrome'
    text: 'Inherited MMR deficiency (Lynch syndrome)'


The "Probable Associated Phenotype" component carries `R210` *and* SNOMED
`716318002` together, on the same `CodeableConcept` - exactly the cross-reference this
notebook had to do manually above, except here it's already been done once, by the
reporting lab, and doesn't need re-deriving by whatever consumes this resource next.

## Updating the EPR: the same `Condition`, now confirmed

`Condition-LynchSyndrome.fsh` - also already published, for this same patient - is
exactly the "after" state this scenario is building towards: `verificationStatus`
moved from `unconfirmed` to `confirmed`, `evidence` pointing at the
`DiagnosticImplication` above. Fetched live, then compared directly against the
"before" `Condition` this notebook opened with:

In [8]:
condition_after = requests.get(f"{IG_BASE}/Condition-c8f82825-e4cb-4e1f-b728-3fd2808e93db.json").json()

before_status = epic_style_condition_before["verificationStatus"]["coding"][0]["code"]
after_status = condition_after["verificationStatus"]["coding"][0]["code"]
print(f"verificationStatus: {before_status} -> {after_status}")
print(f"code: {epic_style_condition_before['code']['coding'][0]['display']!r} (unchanged - same SNOMED concept throughout)")
print(f"evidence: {condition_after.get('evidence', '<none, before>')}")

verificationStatus: unconfirmed -> confirmed
code: 'Lynch syndrome' (unchanged - same SNOMED concept throughout)
evidence: [{'detail': [{'reference': 'Observation/6beb613f-d303-42af-b025-86e8e0872061'}]}]


### Writing it back to Epic - proof of concept only

**Not calling any real Epic endpoint here** - this is illustrative of the shape a
Trust Integration Engine would send to Epic's `Condition` write-back API
([api=949](https://fhir.epic.com/Specifications?api=949), fetch-limitation noted
above), built from the confirmed `Condition` fetched live above. A real, working Epic
connection - SMART Backend Services, JWT-bearer, no client secret - already exists in
[nw-gmsa/julius](https://github.com/nw-gmsa/julius)'s `epic_client.py`, currently
read-only (genomic reports, family history) against Epic's public sandbox; writing a
confirmed `Condition` back like this would be a natural next step from that, not
attempted here.

In [9]:
epic_write_back_payload = {
    "resourceType": "Condition",
    "id": condition_after["id"],
    "clinicalStatus": condition_after["clinicalStatus"],
    "verificationStatus": condition_after["verificationStatus"],
    "category": condition_after["category"],
    "code": condition_after["code"],
    "subject": {"reference": "Patient/EPIC-PATIENT-ID"},  # resolved from the NHS number on the order, not shown here
    "evidence": condition_after["evidence"],
}

print("PUT https://fhir.epic.com/interconnect-fhir-oauth/api/FHIR/R4/Condition/{id}  <- illustrative only, not called")
print(json.dumps(epic_write_back_payload, indent=2))
# requests.put(..., json=epic_write_back_payload, headers={"Authorization": f"Bearer {epic_access_token}"})  # never executed

PUT https://fhir.epic.com/interconnect-fhir-oauth/api/FHIR/R4/Condition/{id}  <- illustrative only, not called
{
  "resourceType": "Condition",
  "id": "c8f82825-e4cb-4e1f-b728-3fd2808e93db",
  "clinicalStatus": {
    "coding": [
      {
        "system": "http://terminology.hl7.org/CodeSystem/condition-clinical",
        "code": "active"
      }
    ]
  },
  "verificationStatus": {
    "coding": [
      {
        "system": "http://terminology.hl7.org/CodeSystem/condition-ver-status",
        "code": "confirmed"
      }
    ]
  },
  "category": [
    {
      "coding": [
        {
          "system": "http://terminology.hl7.org/CodeSystem/condition-category",
          "code": "problem-list-item",
          "display": "Problem List Item"
        }
      ]
    }
  ],
  "code": {
    "coding": [
      {
        "system": "http://snomed.info/sct",
        "code": "716318002",
        "display": "Lynch syndrome"
      }
    ]
  },
  "subject": {
    "reference": "Patient/EPIC-PATIENT-ID"
  

## Why a FHIR supplement, not a `DG1` segment

This notebook's whole report-side worked example - `DiagnosticReportGenomicsReportLS`
above - has no `DG1` segment in it at all; the diagnosis/problem side of this exchange
simply isn't modelled in the current `ORU_R01` shape this IG's examples use. Sending
the confirmed `Condition` as a separate FHIR call to the TIE, alongside the v2
`ORU_R01`, sidesteps that gap rather than closing it - a deliberate choice for this
notebook, not the only one available. A TIE developer could instead add a `DG1`
segment to the `ORU_R01` itself and let the Trust's own v2-side integration carry the
diagnosis update through its existing HL7 v2 problem-list handling - genuinely an
open design question this notebook doesn't resolve, not a settled recommendation.

## Summary

- NHS guidance wants SNOMED CT on `Condition`/`DG1`; Epic is one of the EPRs that
  actually supports it, but the two specific numbered Epic reference pages this
  notebook was pointed at couldn't be fetched live (checked, not assumed) - so the
  `Condition` examples here follow Epic's well-documented general conventions rather
  than quoting Epic's own numbered example verbatim.
- Genomics wants a different code again - a clinical indication/Test Package code, or
  HPO terms for rare disease WGS (confirmed straight off the real, current GMS order
  form PDF: *"HPO terms are NECESSARY ... cannot commence until provided"*) - and
  `13-...ipynb`/`14-...ipynb` both independently found no automated SNOMED CT route to
  either, for a diagnosis code specifically.
- `716318002` "Lynch syndrome" → `R210` "Inherited MMR deficiency (Lynch syndrome)" is
  today's manual answer - still legacy-only, no DGTS `TP` code yet.
- The genomic report's own real, published example for this patient has a free-text
  `conclusion` but no `conclusionCode` - this notebook builds the coded version
  (`ValueSet-GenomicTestOutcomeCodes`, `51968-6` on the `OBX` side) that's missing, and
  shows `DiagnosticImplication` as the more precise, already-modelled alternative that
  carries `R210` and the SNOMED code together on one `CodeableConcept`.
- The "after" `Condition` - `verificationStatus = confirmed`, `evidence` linked to the
  genomic finding - is also already published for this same patient; updating Epic's
  own problem list with it is shown as a payload only, never called, with
  `nw-gmsa/julius`'s real (read-only, for now) Epic connection as the natural next
  step rather than something this notebook attempts.